# Agent 1: 구조 중심성 비교 재현
AI 연구 보조 코드. 실제 조원의 개인 기여가 완료된 것으로 표시하지 않는다.

20개 예비 사례의 입력 → 두 독립 소스 실행 → 출력 검증 → 저장된 공식 웹 CSV와 대조 → 결과 요약 흐름이다. 웹 제출은 이미 실행한 job 5와 7의 원시 CSV를 사용하며, 이 노트북은 웹 재제출이나 공식 채점기를 호출하지 않는다. 라이브 웹 상태·계정 정보 없이 실행 가능하다. 소형 합성 문제 출처는 이 저장소 scenarios/problem-bank, 접근일 2026-11-22이며 protocol.json에 사례 ID와 입력 해시가 있다.

담당 표시는 AI agent 1이다. 실제 제출 전 조원은 코드를 검토·수정·실행하고 본인 담당 범위를 기록해야 한다.

## 데이터 로딩과 고정 소스 확인
작성: AI agent 1

In [1]:
from pathlib import Path
import csv, hashlib, importlib.util, io, json, subprocess, sys
HERE = Path.cwd()
if not (HERE / "protocol.json").exists():
    HERE = HERE / "results/agent-study/2026-11-22/agent-1"
protocol = json.loads((HERE / "protocol.json").read_text())
jobs = json.loads((HERE / "web_jobs.json").read_text())
for job in jobs["jobs"]:
    assert hashlib.sha256((HERE / (job["method"] + ".py")).read_bytes()).hexdigest() == job["source_sha256"]
for item in protocol["inputs"]:
    assert hashlib.sha256((HERE / "inputs" / (item["name"] + ".txt")).read_bytes()).hexdigest() == item["sha256"]
print(len(protocol["cases"]), "fixed inputs and both source hashes verified")

20 fixed inputs and both source hashes verified


## 독립 프로그램 재실행과 공식 제출 출력 대조
작성: AI agent 1. 현재 로컬 실행시간을 과거 웹 측정시간과 혼동하지 않는다.

In [2]:
for method in ("degree", "betweenness"):
    official = {row["scenario"]: row for row in csv.DictReader((HERE / (method + "-web.csv")).open(encoding="utf-8-sig"))}
    for name in protocol["cases"]:
        data = (HERE / "inputs" / (name + ".txt")).read_bytes()
        rows = data.decode().splitlines()
        n, m, d, a = map(int, rows[0].split())
        budget = int(rows[1].split()[0])
        edges = {int(row.split()[0]): list(map(int, row.split())) for row in rows[2+n:2+n+m]}
        output = subprocess.run([sys.executable, str(HERE / (method + ".py"))], input=data, capture_output=True, check=True, timeout=10).stdout
        count, *ids = map(int, output.split())
        assert count == len(ids) and ids == sorted(set(ids))
        assert sum(edges[eid][4] for eid in ids) <= budget
        assert official[name]["status"] == "completed"
        assert ids == list(map(int, official[name]["chosen_edges"].split()))
    print(method, "20/20 deterministic outputs reproduced and matched to web CSV")

degree 20/20 deterministic outputs reproduced and matched to web CSV


betweenness 20/20 deterministic outputs reproduced and matched to web CSV


## 중심성 구현의 독립 검증
작성: AI agent 1. NetworkX는 참고 오라클이며 제출 프로그램의 의존성이 아니다.

In [3]:
import networkx as nx
spec = importlib.util.spec_from_file_location("centrality_submission", HERE / "betweenness.py")
centrality_submission = importlib.util.module_from_spec(spec)
spec.loader.exec_module(centrality_submission)
for name in protocol["cases"]:
    n, edges, budget = centrality_submission.read_problem(io.StringIO((HERE / "inputs" / (name + ".txt")).read_text()))
    graph = nx.Graph()
    graph.add_nodes_from(range(n))
    graph.add_edges_from((u,v) for eid,u,v,capacity,cost in edges)
    expected = nx.edge_betweenness_centrality(graph, normalized=False)
    actual = centrality_submission.centrality(n, edges)
    assert all(abs(float(actual[eid]) - expected[u,v]) < 1e-10 for eid,u,v,capacity,cost in edges)
print("20 independent centrality comparisons passed")

20 independent centrality comparisons passed


## 원시 결과 집계와 해석
작성: AI agent 1. 두 예산은 같은 그래프를 공유하므로20행은10개 그래프 군집이다. 아래 중앙값들은 각각의 기술통계이며 하나의 방호안 점수가 아니다.

In [4]:
spec = importlib.util.spec_from_file_location("analysis", HERE / "analyze.py")
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
summary = analysis.summarize(HERE)
assert summary == json.loads((HERE / "summary.json").read_text())
for method, measures in summary["methods"].items():
    print(method, measures)
print("proposed versus baseline", summary["paired_outcomes"])
print(summary["inference_limit"])

degree {'completed': 20, 'total': 20, 'optimal_score_count': 0, 'median_connected_ratio': 0.6742424242424242, 'median_delivery_ratio': 0.5812629399585922, 'median_search_seconds': 0.0850896745105274, 'max_search_seconds': 0.10846233100164682, 'max_worker_peak_mb': 6.62}
betweenness {'completed': 20, 'total': 20, 'optimal_score_count': 1, 'median_connected_ratio': 0.709040959040959, 'median_delivery_ratio': 0.6071540143705092, 'median_search_seconds': 0.08520792500348762, 'max_search_seconds': 0.13947764100157656, 'max_worker_peak_mb': 6.64}
proposed versus baseline {'win': 8, 'tie': 9, 'loss': 3, 'incomplete': 0}
Two budgets share each graph: 20 rows represent 10 graph clusters, not 20 independent draws. No significance or general superiority claimed.


제안의 예비 비교 결과는 승8·동9·패3이다. 모든 사례에서 개선되는 방식은 아니며 최적 점수 일치도1/20에 불과하다. 두 방법 모두 대략0.1초의 매우 짧은 실행이므로 이20회 측정만으로 속도 차이를 주장하지 않는다. 중심성은 수요·용량·공격 기하를 무시하므로 실제 최악 공격에 중요한 간선을 놓칠 수 있다. 추후 같은 고정 소스를 미사용 시드에 적용하고 그래프 단위로 묶어 효과를 확인한다.

## 관련 연구와 사람의 검토
Freeman (1977), Sociometry 40(1):35–41, DOI 10.2307/3033543. 매개 중심성을 구조적 중요도로 해석하는 출발점이며, 원 논문의 정점 중심성이 곧 방호 최적성이라는 뜻은 아니다.

Brandes (2001), Journal of Mathematical Sociology 25(2):163–177, DOI 10.1080/0022250X.2001.9990249. BFS 최단경로 수와 의존도 누적으로 비가중 중심성을 계산한다. Fraction을 사용하는 실제 비트 연산 비용은 단순 O(VE) 연산 수 이상일 수 있다. 원출처 URL은 references.json에 있다.

본인은 왜 최단경로 개수를 나누는지, 무방향에서 왜2로 나누는지, 비용 탐욕법이 방호 최적성을 보장하지 않는 이유를 설명하고 작은 예제를 직접 실행해야 한다. AI 작성 자료를 본인이 직접 작성한 것으로 표시하지 않는다.